This notebook shows how to create a cache. If you already have downloaded a cache, you don't need to run this notebook.

In [ ]:
from pathlib import Path
from datasets import load_dataset
from vectormesh import Vectorizer

We get the IMDB dataset from huggingface.

In [ ]:
dataset = load_dataset("stanfordnlp/imdb", cache_dir="../assets/imdb/")
dataset_tag = "imdb"
dataset

This gives us 25.000 train and 25.000 test movie reviews, that can be labeled positive or negative.

In [ ]:
train = dataset["train"]
train

I load the aktes dataset from the assets folder, and set up a vectorizer with a huggingface model.
You can find more information about the models on huggingface:

- [granite](https://huggingface.co/ibm-granite/granite-embedding-small-english-r2) 
- [bge](https://huggingface.co/BAAI/bge-small-en-v1.5)
- [MiniLM-L6](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)


In [ ]:
model_name = "ibm-granite/granite-embedding-small-english-r2"
# look up other model tags on huggingface to change the model, eg
# model_name = "sentence-transformers/all-MiniLM-L6-v2" 
tag = model_name.split("/")[-1]
vectorizer = Vectorizer(model_name=model_name, col_name=tag, max_length=512)

Some models have a bigger context window. However, using big context windows (eg 8192) typically takes up a lot of RAM most machines don't have. So we limit the context of a single chunk to 512.

Lets use a sample, because vectorization can take a while on the full dataset and this is just a demo.

In [ ]:
sample = train.select(range(64))

The VectorCache will 
- loop over the full dataset (in this notebook, the sample. For the full dataset, use hardware accelaration.)
- add metadata to keep track of which vectors belong to which dataset and vectorizer
- store the vectorcache on disk

In [ ]:
from vectormesh import VectorCache

vectorcache = VectorCache.create(
    cache_dir=Path("tmp/artefacts"),
    vectorizer=vectorizer,
    dataset=sample,
    dataset_tag=dataset_tag,
)

This will take a while for the full dataset. The core idea of the VectorCache is, that someone can run this process once, and then you can reuse the VectorCache to train a small architecture on top of the vectors. Therefore, in other notebooks we will load the vectorcache from disk.

Check the vectorcache for some usefull metadata:

In [ ]:
vectorcache.metadata

```json
{'granite-embedding-small-english-r2': {'vectormesh_version': '0.1.0',
  'model_tag': 'ibm-granite/granite-embedding-small-english-r2',
  'vectorizer_type': 'Vectorizer',
  'tensordtype': 2,
  'hidden_size': 384,
  'context_size': 512,
  'chunk_sizes': Counter({1: 58, 2: 4, 3: 2})},
 'features': ['text', 'label', 'granite-embedding-small-english-r2'],
 'created_at': '2026-06-06T10:03:17.309144',
 'num_observations': 64}
```

Every document is split into chunks of max 512 tokens, with an overlap of about 50 tokens (512 // 10).
This means that every document is turned into a 2D tensor with shape (chunks, dim) where dim is the embedding dimension (eg 768 for bert-base-uncased)

If we run `.__get_item__` on the vectorcache, by doing `vectorcache[0]`, this is being passed on to the underlying dataset, and we get the first document in the dataset.

- "text" is the original text
- "label" or "labels" is the thing we want to predict
- we have an additional column with the vector we produced with the embedding model.

In [ ]:
vectorcache[0]

```
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that ...',
 'label': tensor(0),
 'granite-embedding-small-english-r2': tensor([[ 1.3906e+00, -1.4160e-01, -4.8242e-01,  3.6328e-01,  6.7578e-01,
          -9.6484e-01, -5.1172e-01,  7.7734e-01,  7.5195e-02, -9.6484e-01,
          ...
          -7.4219e-01, -6.2256e-02,  1.0312e+00,  8.7891e-01, -5.1953e-01,
           4.6289e-01,  9.9609e-01, -7.8125e-01, -8.3203e-01]])}
```

We can confirm the shape of the vector

In [ ]:
vectorcache[0][tag].shape


We can now **extend** the existing dataset with more vectors! 

I created a RegexVectorizer, that turn regex-matches into a binary vector. There are regexes for words like "good", "bad" etc, and this is turned into a binary vector `[0, 0, 1, 1, 0, ..]` when these "regex features" are present in the text.

You can run it on the full trainset, it is pretty fast.

from the imdb dataset, i created `build_imdb_review_pattern` and `harmonize_imdb_match`, for the legal data there are `build_legal_references_pattern` and `harmonize_legal_reference` functions. You can find all of them in `vectormesh/data/vectorizers.py`.

In [ ]:
from vectormesh import RegexVectorizer
from vectormesh.data.vectorizers import (
    build_imdb_review_pattern,
    harmonize_imdb_match
)


# Initialize & fit with training_texts
regexvectorizer = RegexVectorizer(
    pattern_builder=build_imdb_review_pattern,
    harmonizer=harmonize_imdb_match,
    min_doc_frequency=15,
    max_features=200,
    device="cpu",
    training_texts=train["text"],  # fit it on the full train set
)
regexvectorizer.get_metadata

```json
{'model_name': 'regex_vectorizer',
 'col_name': 'regex_features',
 'hidden_size': 43,
 'context_size': None,
 'min_doc_frequency': 15,
 'max_features': 200}
```

It initialized on the full imdb trainset, and found 43 features. This is determined by

1. min_doc_frequency: the minimum amount of documents a feature must appear in
2. max_features: the maximum amount of features to be extracted

To have an idea what it found, we can print some stats:

In [ ]:
regexvectorizer.print_stats()

![](img/regex_stats.png)

But this was just the "training" of the regexvectorizer. We didnt actually store this as data in the vectorcache yet. We can do this now.

First, lets pick up the existing vectorcache from disk, such that we can extend it with the vectors.

In [ ]:
new_tag = next(
    Path("tmp/artefacts").glob("*imdb*/")
)  # next picks the first one it finds
new_tag.name

This is the foldername of the dataset in `tmp/artefacts` we want to extend.

For the new vectors, the cache will use the `.col_name` from `regexvectorizer`, which you can change.

In [ ]:
regexvectorizer.col_name

We can now:
- use the existing vectorcache dataset with the huggingface vectors
- extend it with the regex vectors
- regexvectorizer has a feature `col_name`, which is set to `regex_features` by default. You can change it if you want, for example when you modify the regexes.
- save the new cache to tmp/artefacts

In [ ]:
updated_cache = VectorCache.create(
    cache_dir=Path("tmp/artefacts"),
    vectorizer=regexvectorizer,  # use our new regex vectorizer
    dataset=vectorcache.dataset,  # use the existing dataset
    dataset_tag=new_tag.name,  # this will check for existing metadata.json in the old folder
)

The data is now extended with the regex vectors!
Lets check the metadata:

In [ ]:
updated_cache.metadata

```json
{'granite-embedding-small-english-r2': {'vectormesh_version': '0.1.0',
  'model_tag': 'ibm-granite/granite-embedding-small-english-r2',
  'vectorizer_type': 'Vectorizer',
  'tensordtype': 2,
  'hidden_size': 384,
  'context_size': 512,
  'chunk_sizes': {}},
 'features': ['text',
  'label',
  'granite-embedding-small-english-r2',
  'regex_features'],
 'created_at': '2026-06-05T21:42:58.361876',
 'num_observations': 64,
 'regex_features': {'vectormesh_version': '0.1.0',
  'model_tag': 'regex_vectorizer',
  'vectorizer_type': 'RegexVectorizer',
  'tensordtype': 1,
  'hidden_size': 43,
  'context_size': None,
  'chunk_sizes': None}}
```

As you can see, we now have:

1. 2d Tensors: vectors with size (chunks x dim), where dim is 384 from our huggingface embedding model and every document is split in chunks of 512 tokens.
2. 1d Tensors: regex-vectors, that are one-hot vectors, with a size of 43

lets have a look at an actual observation:

In [ ]:
updated_cache[0]

```json
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of ...',
 'label': tensor(0),
 'granite-embedding-small-english-r2': tensor([[ 1.3906e+00, -1.4160e-01, -4.8242e-01,  3.6328e-01,  6.7578e-01,
          -9.6484e-01, -5.1172e-01,  7.7734e-01,  7.5195e-02, -9.6484e-01,
                                ...
          -7.4219e-01, -6.2256e-02,  1.0312e+00,  8.7891e-01, -5.1953e-01,
           4.6289e-01,  9.9609e-01, -7.8125e-01, -8.3203e-01]]),
 'regex_features': tensor([1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
         0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.])}
```

You see:
- The original text
- The text, embedded as a 2D tensor (chunks x dim)
- the text, encoded with the regexvectorizer, as a binary 1D tensor

Confirm the shape of the vectors:

In [ ]:
print(f"shape of the embedding: {updated_cache[0][tag].shape}") 
print(f"shape of the regex features: {updated_cache[0]['regex_features'].shape}")

Lets clean up the tmp/artefacts folder, because we only made it for a small sample

In [ ]:
import shutil

shutil.rmtree("tmp/", ignore_errors=True)
shutil.rmtree("logs/", ignore_errors=True)
